In [11]:
import sys
print('Kernel Python:', sys.executable)
%pip install numpy pandas ipython ipywidgets

Kernel Python: C:\Users\HP\AppData\Local\Programs\Python\Python311\python.exe
Note: you may need to restart the kernel to use updated packages.


# Final UI Notebook (Pipeline Controller)

Notebook này là giao diện chạy end-to-end cho dự án.

- UI và luồng thao tác nằm ở đây.
- Các file trong modules chỉ đóng vai trò backend functions.

Quy trình:
1. Setup môi trường + import module
2. Chọn cấu hình bằng widget
3. Load dữ liệu
4. Run pipeline và xem bảng kết quả

In [12]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / "modules").exists() and (p / "requirements.txt").exists():
            return p
    raise RuntimeError("Cannot find project root")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from modules.config import Config
from modules.data_loader import load_data
from modules.pipeline import build_features, run_evaluation

base_cfg = Config()
state = {
    "cfg": None,
    "result_df": None,
    "x_train": None,
    "x_test": None,
    "n_train": None,
    "n_test": None,
}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PROJECT_ROOT: D:\HCMUT\Nam 3\ky2\ML\Assignment1\official\MachineLearning_TextModule


## UI controls

Chọn mode/feature/model ngay trong notebook UI.

In [13]:
import ipywidgets as widgets
from IPython.display import display

mode_w = widgets.Dropdown(
    options=["demo", "full"],
    value=base_cfg.mode,
    description="Mode:",
    layout=widgets.Layout(width="220px"),
)

feature_w = widgets.Dropdown(
    options=["tfidf", "sbert"],
    value=base_cfg.feature_method,
    description="Feature:",
    layout=widgets.Layout(width="220px"),
)

model_w = widgets.Dropdown(
    options=["logistic_regression", "svm", "naive_bayes"],
    value=base_cfg.model_type,
    description="Model:",
    layout=widgets.Layout(width="280px"),
)

preprocess_w = widgets.Checkbox(
    value=base_cfg.use_preprocessing,
    description="Use preprocessing",
)

run_btn = widgets.Button(
    description="Run pipeline",
    button_style="success",
    tooltip="Build features + train + evaluate",
)

out = widgets.Output()

display(widgets.HBox([mode_w, feature_w, model_w]))
display(preprocess_w)
display(run_btn)
display(out)

Checkbox(value=True, description='Use preprocessing')

Button(button_style='success', description='Run pipeline', style=ButtonStyle(), tooltip='Build features + trai…

Output()

## Load dataset (from module backend)

Dữ liệu được tải qua modules.data_loader để thống nhất source of truth.

In [ ]:
train_texts, train_labels, test_texts, test_labels, info = load_data(base_cfg.dataset_name)

print(info)
print("Train:", len(train_texts), "Test:", len(test_texts))
print("Sample text:", str(train_texts[0])[:160], "...")

## Run pipeline from UI

Cell này gắn sự kiện cho nút Run và hiển thị bảng metrics ngay trong notebook.

In [ ]:
def build_cfg_from_ui():
    return replace(
        base_cfg,
        mode=mode_w.value,
        feature_method=feature_w.value,
        model_type=model_w.value,
        use_preprocessing=preprocess_w.value,
    )

def on_run_click(_):
    out.clear_output()
    with out:
        cfg = build_cfg_from_ui()
        print("Running config:")
        print(f"- mode={cfg.mode}")
        print(f"- feature_method={cfg.feature_method}")
        print(f"- model_type={cfg.model_type}")
        print(f"- use_preprocessing={cfg.use_preprocessing}")

        X_train, X_test, n_train, n_test = build_features(cfg, train_texts, test_texts)
        result, df_metrics = run_evaluation(
            X_train,
            train_labels,
            X_test,
            test_labels,
            cfg,
            method_name=f"{cfg.feature_method}+{cfg.model_type}",
        )

        state["cfg"] = cfg
        state["x_train"] = X_train
        state["x_test"] = X_test
        state["n_train"] = n_train
        state["n_test"] = n_test
        state["result_df"] = df_metrics
        state["eval_result"] = result

        tracker = ExperimentTracker(Path("../results/experiments"))
        tracker.log_run(
            config=cfg,
            metrics=result,
            predictions=None, 
            artifact_paths={"confusion_matrix": "cm.png"}
        )
        print(f"Experiment logged. Total runs: {len(tracker.runs)}")

        print(f"Used samples: train={n_train}, test={n_test}")
        print(f"Feature shapes: train={X_train.shape}, test={X_test.shape}")
        display(df_metrics)
run_btn.on_click(on_run_click)
print("UI callback is ready. Click 'Run pipeline' above.")

## Manual trigger (optional)

Dùng cell này nếu bạn muốn chạy nhanh mà không bấm nút.

In [ ]:
# Optional: run once from current UI values
on_run_click(None)

## Output artifacts

Kiểm tra file output theo feature method đang chạy.

In [ ]:
cfg = state.get("cfg")
if cfg is None:
    print("Chưa có kết quả. Hãy chạy pipeline trước.")
else:
    print("Last config:", cfg)
    if cfg.feature_method == "tfidf":
        print("TF-IDF train path:", cfg.tfidf_train_path)
        print("TF-IDF test path:", cfg.tfidf_test_path)
        print("Exists:", cfg.tfidf_train_path.exists(), cfg.tfidf_test_path.exists())
    else:
        print("SBERT train path:", cfg.bert_train_path)
        print("SBERT test path:", cfg.bert_test_path)
        print("Exists:", cfg.bert_train_path.exists(), cfg.bert_test_path.exists())

    if state.get("result_df") is not None:
        display(state["result_df"])

## Week 3 validation checklist

Cell này kiểm tra nhanh các artifacts bắt buộc của Tuần 3:
- Embedding theo từng scale benchmark (`5k_2k`, `20k_2k`)
- Bảng báo cáo `results/tables/bert_benchmark_results.csv`

Nếu thiếu, chạy 2 cell code ngay dưới để tạo lại đầy đủ.

In [ ]:
import subprocess
import sys

# Đặt True khi bạn muốn chạy benchmark Tuần 3 đầy đủ.
RUN_FULL_WEEK3_BENCHMARK = False

if RUN_FULL_WEEK3_BENCHMARK:
    cmd = [sys.executable, "bert_benchmark.py"]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=str(PROJECT_ROOT), check=False)
    print("Exit code:", completed.returncode)
else:
    print("Skip run. Set RUN_FULL_WEEK3_BENCHMARK = True để chạy benchmark Tuần 3.")

In [ ]:
from pathlib import Path

project_root = find_project_root()
required_paths = [
    project_root / "features" / "bert" / "5k_2k" / "bert_train.npy",
    project_root / "features" / "bert" / "5k_2k" / "bert_test.npy",
    project_root / "features" / "bert" / "20k_2k" / "bert_train.npy",
    project_root / "features" / "bert" / "20k_2k" / "bert_test.npy",
    project_root / "results" / "tables" / "bert_benchmark_results.csv",
]

missing = [p for p in required_paths if not p.exists()]

print("Week 3 artifact check:")
for p in required_paths:
    print(f"- {p.relative_to(project_root)}: {'OK' if p.exists() else 'MISSING'}")

if missing:
    print("\n=> Chưa đủ artifacts Tuần 3. Chạy cell benchmark bên dưới để tạo lại.")
else:
    print("\n=> Đã đủ artifacts Tuần 3.")